In [2]:
import os
import pandas as pd
import numpy as np
import kagglehub
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input, TextVectorization

# First dataset
print("Downloading first dataset (spam-email-dataset)...")
path1 = kagglehub.dataset_download("jackksoncsie/spam-email-dataset")
csv_files1 = [f for f in os.listdir(path1) if f.endswith('.csv')]
csv_path1 = os.path.join(path1, csv_files1[0])
df1 = pd.read_csv(csv_path1)

# Standardize column names
df1 = df1.rename(columns={'text': 'text', 'spam': 'spam'})
df1 = df1[['text', 'spam']]
print(f"Dataset 1 shape: {df1.shape}")

# Second dataset
print("\nDownloading second dataset (email-spam-classification-dataset)...")
path2 = kagglehub.dataset_download("purusinghvi/email-spam-classification-dataset")
csv_files2 = [f for f in os.listdir(path2) if f.endswith('.csv')]
csv_path2 = os.path.join(path2, csv_files2[0])
df2 = pd.read_csv(csv_path2)

# Identify columns dynamically
text_col_df2 = 'email' if 'email' in df2.columns else df2.columns[1]
label_col_df2 = 'label' if 'label' in df2.columns else df2.columns[0]

df2_cleaned = pd.DataFrame()
df2_cleaned['text'] = df2[text_col_df2].astype(str)

# Standardize string labels ('spam' / 'ham') to binary numeric values
if df2[label_col_df2].dtype == object:
    df2_cleaned['spam'] = df2[label_col_df2].str.lower().map({'spam': 1, 'ham': 0})
else:
    df2_cleaned['spam'] = df2[label_col_df2].astype(int)

df2_cleaned = df2_cleaned.dropna().reset_index(drop=True)
print(f"Dataset 2 standardized shape: {df2_cleaned.shape}")

# third dataset
print("\nDownloading third dataset (spam-assassin-email-classification-dataset)...")
try:
    # Download the dataset using download instead of load_dataset to find files dynamically
    path3 = kagglehub.dataset_download("ganiyuolalekan/spam-assassin-email-classification-dataset")

    csv_files3 = []
    for root, dirs, files in os.walk(path3):
        for file in files:
            if file.endswith('.csv'):
                csv_files3.append(os.path.join(root, file))

    if csv_files3:
        print(f"Found CSV file: {csv_files3[0]}")
        df3 = pd.read_csv(csv_files3[0])

        # Determine correct text and label columns dynamically
        text_col_df3 = 'text' if 'text' in df3.columns else ('body' if 'body' in df3.columns else df3.columns[0])
        label_col_df3 = 'target' if 'target' in df3.columns else ('spam' if 'spam' in df3.columns else df3.columns[-1])

        df3_cleaned = pd.DataFrame()
        df3_cleaned['text'] = df3[text_col_df3].astype(str)
        # Convert target label cleanly
        df3_cleaned['spam'] = df3[label_col_df3].astype(int)
        df3_cleaned = df3_cleaned.dropna().reset_index(drop=True)
        print(f"Dataset 3 standardized shape: {df3_cleaned.shape}")
    else:
        print("No CSV file found in the downloaded dataset archive.")
        df3_cleaned = pd.DataFrame(columns=['text', 'spam'])

except Exception as e:
    print(f"Failed to load the third dataset: {e}")
    df3_cleaned = pd.DataFrame(columns=['text', 'spam'])

# merge datasets, shuffle them, split into training and testing sets
combined_df = pd.concat([df1, df2_cleaned, df3_cleaned], ignore_index=True)
combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)
print(f"\nCombined Dataset (All 3 sources) shape: {combined_df.shape}")
print(combined_df['spam'].value_counts())

X_comb = combined_df['text'].astype(str).values
y_comb = combined_df['spam'].astype(np.int32).values

# Re-split: 70% Train, 15% Validation, 15% Test
X_train, X_temp, y_train, y_temp = train_test_split(
    X_comb, y_comb, test_size=0.30, random_state=42, stratify=y_comb
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"\nNew Splits:")
print(f"- Train size: {len(X_train)}")
print(f"- Validation size: {len(X_val)}")
print(f"- Testing size: {len(X_test)}")

# Model traingin with text vectorization
max_features = 20000

tf_idf_vectorizer = TextVectorization(
    max_tokens=max_features,
    output_mode='tf-idf',
    ngrams=(1, 2)
)

print("\nAdapting TF-IDF Vectorizer on expanded training data...")
tf_idf_vectorizer.adapt(X_train)

improved_model = Sequential([
    Input(shape=(1,), dtype=tf.string),
    tf_idf_vectorizer,
    Dense(64, activation='relu'),
    Dropout(0.4),
    Dense(16, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

improved_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("\nTraining the improved TF-IDF model...")
history_improved = improved_model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_data=(X_val, y_val)
)

# Evaluate performance on test set
imp_loss, imp_accuracy = improved_model.evaluate(X_test, y_test, verbose=0)
print(f"\n--- Model Performance on Testing Set ---")
print(f"Test Loss: {imp_loss:.4f}")
print(f"Test Accuracy: {imp_accuracy*100:.2f}%")

Using Colab cache for faster access to the 'spam-email-dataset' dataset.
Dataset 1 shape: (5728, 2)

Using Colab cache for faster access to the 'email-spam-classification-dataset' dataset.
Dataset 2 standardized shape: (83448, 2)



100%|██████████| 7.28M/7.28M [00:00<00:00, 92.8MB/s]

Extracting files...


Found CSV file: /root/.cache/kagglehub/datasets/ganiyuolalekan/spam-assassin-email-classification-dataset/versions/1/spam_assassin.csv
Third dataset preview:


,text,target
0,From ilug-admin@linux.ie Mon Jul 29 11:28:02 2...,0
1,From gort44@excite.com Mon Jun 24 17:54:21 200...,1


Dataset 3 standardized shape: (5796, 2)

Combined Dataset (All 3 sources) shape: (94972, 2)
spam
0    47798
1    47174
Name: count, dtype: int64

New Splits:
- Train size: 66480
- Validation size: 14246
- Testing size: 14246

Adapting TF-IDF Vectorizer on expanded training data...

Training the improved TF-IDF model...
Epoch 1/5
1039/1039 ━━━━━━━━━━━━━━━━━━━━ 37s 34ms/step - accuracy: 0.9493 - loss: 0.1781 - val_accuracy: 0.9871 - val_loss: 0.0620
Epoch 2/5
1039/1039 ━━━━━━━━━━━━━━━━━━━━ 35s 34ms/step - accuracy: 0.9842 - loss: 0.0728 - val_accuracy: 0.9888 - val_loss: 0.0442
Epoch 3/5
1039/1039 ━━━━━━━━━━━━━━━━━━━━ 36s 34ms/step - accuracy: 0.9878 - loss: 0.0548 - val_accuracy: 0.9898 - val_loss: 0.0468
Epoch 4/5
1039/1039 ━━━━━━━━━━━━━━━━━━━━ 35s 34ms/step - accuracy: 0.9907 - loss: 0.0447 - val_accuracy: 0.9903 - val_loss: 0.0419
Epoch 5/5
1039/1039 ━━━━━━━━━━━━━━━━━━━━ 42s 35ms/step - accuracy: 0.9938 - loss: 0.0333 - val_accuracy: 0.9900 - val_loss: 0.0400

--- Model Performance o

In [4]:
#@title Interactive Spam Classifier
# Run this cell to test custom email transcripts against the trained Keras model.

import numpy as np

def predict_emails(emails):
    if not emails:
        print("No valid inputs provided.")
        return

    for idx, email_text in enumerate(emails, 1):
        # Determine if there is a Subject ending with "  "
        subject_display = email_text
        if "  " in email_text:
            subject_part = email_text.split("  ")[0]
            subject_display = f"{subject_part} (subject message ends with \"  \")"

        # Predict using the trained TF-IDF model
        input_data = np.array([email_text], dtype=object)
        prediction = improved_model.predict(input_data, verbose=0)[0][0]

        # Print to match requested output format
        print(f"\n-- start of prediction #{idx}")
        print("="*43)
        print(f"Subject: {subject_display}")
        print("="*43)
        if prediction >= 0.955:
            print(f"Prediction: SPAM (Confidence: {prediction * 100:.2f}%)")
        else:
            print(f"Prediction: NOT SPAM (Confidence: {(1 - prediction) * 100:.2f}%)")
        print("="*43)
        print(f"-- end of this prediction #{idx} --")

# Interactive loop accepting lines dynamically
print("Enter your email transcripts below (one email per line).")
print("To finish entering and view results, simply press ENTER on an empty line:\n")

user_emails = []
while True:
    line = input(f"Email #{len(user_emails)+1}: ").strip()
    if not line:
        break
    user_emails.append(line)

predict_emails(user_emails)

Enter your email transcripts below (one email per line).
To finish entering and view results, simply press ENTER on an empty line:

Email #1: Subject: Folder shared with you: "Databases Final Project"  Gideon Mallinson shared a folder Header profile photo Gideon Mallinson (gideon.mallinson@eagles.oc.edu) has invited you to contribute to the following shared folder:
Email #2: Subject: Last chance for a $20 Brew gift card! (You guys please we are so desperate for responses! PLEASE HELP US!! 😭)  GUYS, PLEASE HELP US PLEASE, we’re Marketing Research students and WE NEED YOUR RESPONSES!!! We are working with our client, FNB Community Bank, to learn more about the banking habits of young people. Please fill out this MERE 5 MINUTE SURVEY. All responses will be anonymous, with answers only shared between us, our professor, Dr. Roper, and our client, FNB. If you answer by MIDNIGHT on November 6th (TOMORROW!), you can enter into a drawing to win a $20 Brew gift card! We PROMISE we are actual stu